# Rentabilidad y política de descuentos en Superstore
**Visualización de Datos (TIEL16) · EVA2 · Unidad 2 · INACAP**
Diego Martínez González, Analista Programador

Dataset: [Sample - Superstore (Kaggle)](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final).
Este cuaderno muestra los datos limpios, las 5 visualizaciones y su interpretación de negocio.
El código de limpieza y de los gráficos está en el repositorio
([MargonDiego/eva2-superstore](https://github.com/MargonDiego/eva2-superstore), carpeta `src/`).
Para verlo completo, ejecuta todas las celdas (Colab: *Entorno de ejecución → Ejecutar todo*; Databricks: *Run all*).

## 0. Carga del código y los datos

In [ ]:
%pip install -q seaborn

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

REPO = Path(tempfile.gettempdir()) / "eva2-superstore"
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/MargonDiego/eva2-superstore.git", str(REPO)],
                   check=True)
sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

import graficos, limpieza, textos

graficos.configurar_estilo()
pd.set_option("display.max_colwidth", 200, "display.width", 200)


def md(texto):
    # el signo $ se escapa para que el cuaderno no lo lea como fórmula
    display(Markdown(texto.replace("$", chr(92) + "$")))

## 1. Contexto

In [ ]:
crudo = limpieza.cargar_crudo()
df, bitacora, resumen = limpieza.limpiar()
fichas = textos.ficha_visualizaciones(df)
plt.close("all")

md(textos.CONTEXTO["descripcion"])
md("**Pregunta central.** " + textos.CONTEXTO["pregunta_central"])
print(f"Líneas de pedido: {len(df):,} | Ventas: {graficos._usd(df.Sales.sum())} | "
      f"Utilidad: {graficos._usd(df.Profit.sum())} | Margen: {graficos._pct(df.Profit.sum() / df.Sales.sum())}")
print("Periodo:", resumen["periodo"])
pd.DataFrame({"Tipo": [f["tipo"] for f in fichas], "Pregunta": [f["pregunta"] for f in fichas],
              "Gráfico": [f["grafico"] for f in fichas]}, index=range(1, len(fichas) + 1))

## 2. Preparación de datos
Revisé integridad, calidad y homologación. Cada problema quedó en una bitácora con la decisión tomada y su justificación.

In [ ]:
print(f"Filas originales: {resumen['filas_iniciales']:,} | finales: {resumen['filas_finales']:,} | "
      f"nulos en el original: {int(crudo.isna().sum().sum())} | decisiones documentadas: {len(bitacora)}")
bitacora

In [ ]:
# Registros por región antes y después de limpiar
comparacion = pd.DataFrame({"Antes": crudo["Region"].value_counts(), "Después": df["Region"].value_counts()})
comparacion["Diferencia"] = comparacion["Después"] - comparacion["Antes"]
comparacion

In [ ]:
# Código de limpieza (src/limpieza.py)
import inspect
print(inspect.getsource(limpieza.limpiar))

## 3. Visualizaciones e interpretación de negocio

In [ ]:
for i, f in enumerate(fichas, 1):
    md(f"### 3.{i} {f['tipo']}: {f['pregunta']}")
    f["funcion"](df)
    plt.show()
    md(f"**Qué muestra.** {f['lectura']}")
    md(f"**Decisión que apoya.** {f['decision']}")
    md(f"**Por qué este gráfico.** {f['por_que']}")
    md(f"**Configuración.** {f['configuracion']}")

## 4. Decisiones con implicancia ética

In [ ]:
atipicos = limpieza.efecto_eliminar_atipicos(df)
for i, e in enumerate(textos.reflexion_etica(df, atipicos, resumen), 1):
    md(f"#### 4.{i} {e['decision']}")
    md(f"**Situación.** {e['contexto']}")
    md(f"**Consecuencia.** {e['consecuencia']}")
    if i == 1:
        display(atipicos.round(3))
    if i == 2:
        display(pd.Series(resumen["estados_muestra_baja"], name="Líneas de pedido").to_frame())

## 5. Conclusiones

In [ ]:
for c in textos.conclusiones(df):
    md("- " + c)

## 6. Exploración libre
Filtra por región y año. Las interpretaciones del informe se basan en el dataset completo.

In [ ]:
import ipywidgets as widgets


def explorar(regiones, anios):
    sub = df[df["Region"].isin(regiones) & df["Año"].isin(anios)]
    if sub.empty:
        print("No hay datos con esos filtros.")
        return
    print(f"Ventas: {graficos._usd(sub.Sales.sum())} | Utilidad: {graficos._usd(sub.Profit.sum())} | "
          f"Margen: {graficos._pct(sub.Profit.sum() / sub.Sales.sum(), 1)}")
    fig1, _ = graficos.g1_utilidad_subcategoria(sub)
    fig1.axes[0].set_title("Utilidad por subcategoría")
    fig2, _ = graficos.g3_descuento_margen(sub)
    fig2.axes[0].set_title("Descuento vs. margen")
    plt.show()


todas_regiones, todos_anios = sorted(df["Region"].unique()), sorted(df["Año"].unique())
widgets.interact(
    explorar,
    regiones=widgets.SelectMultiple(options=todas_regiones, value=tuple(todas_regiones), description="Región"),
    anios=widgets.SelectMultiple(options=todos_anios, value=tuple(todos_anios), description="Año"),
)